# QuickTalk: chat-only re-run (Colab, A100, ~15 min)

Re-does ONLY the chat training of run 7, from run 7's finished pretraining (`MyDrive/quicktalk_run7/checkpoints/pretrain_final.pt`).
Nothing is downloaded and there is no pretraining. Output: `MyDrive/quicktalk_run7b/` and `MyDrive/quicktalk_run7b_model.zip`.

**Run 7b chat mix** (run 7 test: stories 14%, multiturn 24%, multi-question 5%; behaviour + maths were 68% of chat training):
- maths removed (`--cap math=0`; arithmetic needs a digit-level tokenizer first), behaviour Q&A capped at 15,000 of 45,804
- conversation patterns 4x (was 3x), 3 passes (was 2), learning rate 1.5e-4 (was 1e-4)
- result: about 72% conversation patterns, 28% behaviour; the chat eval set is unchanged, so eval loss compares with run 7 (2.168)

*Runtime > Change runtime type > A100*, then *Runtime > Run all*.


In [ ]:
# 1. Check the GPU: A100 only
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > A100 GPU'
name = torch.cuda.get_device_name(0); print('GPU:', name)
assert 'A100' in name, f'This runtime has {name}, not an A100. Change the runtime type to A100 and run again.'

In [ ]:
# 2. Drive: new folder for this run; copy the tokenizer and run 7's pretrained weights into it
import os, shutil
RUN, SRC_RUN = 'quicktalk_run7b', 'quicktalk_run7'
CHAT_REPEAT, SFT_EPOCHS, SFT_LR, SHAPE = 4, 3, 1.5e-4, '768,12,12'
CAPS = '--cap behaviour=15000 --cap math=0'
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'; WORK = f'{BASE}/{RUN}'; SRC = f'{BASE}/{SRC_RUN}'
assert os.path.exists(f'{SRC}/checkpoints/pretrain_final.pt'), f'{SRC}/checkpoints/pretrain_final.pt not found (run 7 folder)'
os.makedirs(f'{WORK}/checkpoints', exist_ok=True)
for f in ('tokenizer.json', 'checkpoints/pretrain_final.pt'):
    if not os.path.exists(f'{WORK}/{f}'): shutil.copy(f'{SRC}/{f}', f'{WORK}/{f}'); print('copied', f)
print('work dir:', WORK, sorted(os.listdir(WORK)))

In [ ]:
# 3. Get the code and the chat data (sparse clone: only train/build and train/colab)
import subprocess
REPO, BRANCH, CODE = 'sraivante/quicktalk', 'claude/intelligent-ride-oxqjfy', '/content/quicktalk'
if not os.path.exists(f'{CODE}/train/build/manifest.json'):
    subprocess.run(['rm', '-rf', CODE])
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--filter=blob:none', '--sparse',
                    f'https://github.com/{REPO}.git', CODE], check=True)
    subprocess.run(['git', '-C', CODE, 'sparse-checkout', 'set', 'train/build', 'train/colab'], check=True)
else:
    subprocess.run(['git', '-C', CODE, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', CODE, 'reset', '--hard', 'FETCH_HEAD'], check=True)
LM, DATA = f'{CODE}/train/colab/quicktalk_lm.py', f'{CODE}/train/build'
!pip -q install tokenizers

In [ ]:
# 4. Chat token files (training set capped and upsampled; eval set unchanged) and the plan (~1 min)
!python {LM} prepare --data {DATA} --workdir {WORK} --sft-only --chat-repeat {CHAT_REPEAT} {CAPS}
!python {LM} plan --workdir {WORK} --pretrain-epochs 0 --sft-epochs {SFT_EPOCHS} --dropout 0 --shape {SHAPE}

In [ ]:
# 5. Chat training (~5-8 min). Safe to re-run: it resumes from the last checkpoint.
subprocess.run(f'rsync -a {WORK}/data/ /content/cache/data/', shell=True, check=True)
!python {LM} train --workdir {WORK} --data-dir /content/cache/data --stage sft --batch 64 --lr {SFT_LR} --ckpt-every 200 2>&1 | tee -a {WORK}/train.log

In [ ]:
# 6. Talk to the model: run 7 and run 7b on the same questions
PROMPTS = ' '.join(f'--prompt "{q}"' for q in [
    'Hi! How was your day?',
    'What should I do if my friend is upset with me?',
    'Correct this: She dont like apples.',
    'What does the word diligent mean?',
    'Kal mera exam hai, I am very nervous. What should I do?',
    'Write a short note to thank my teacher.',
    '1. What is a synonym for happy? 2. Correct this: He go to school yesterday.',
    'What will the weather be in my town tomorrow?',
    'Write a short story about a lost puppy who finds its way home.',
    'Rewrite in the past tense: I walk to school and meet my friends.',
])
for run in (SRC_RUN, RUN):
    print(f'\n===== {run} =====')
    !python {LM} chat --workdir {BASE}/{run} {PROMPTS} --temperature 0.6

In [ ]:
# 7. Finish: zip the model to Drive, verify it, then disconnect this runtime (frees the A100)
import zipfile, glob
from google.colab import runtime
CK = f'{WORK}/checkpoints/sft_final.pt'
assert os.path.exists(CK), 'chat training has not finished: re-run cell 5'
ZIP = f'{BASE}/{RUN}_model.zip'
items = [CK, f'{WORK}/tokenizer.json', f'{WORK}/plan.json', LM] + glob.glob(f'{WORK}/log_*.csv') + [f'{WORK}/train.log']
with zipfile.ZipFile('/content/model.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in items:
        if os.path.exists(f): z.write(f, os.path.basename(f))
assert zipfile.ZipFile('/content/model.zip').testzip() is None, 'zip is corrupt'
shutil.copy('/content/model.zip', ZIP)
ok = os.path.getsize(ZIP) == os.path.getsize('/content/model.zip') and zipfile.ZipFile(ZIP).testzip() is None
print('zip saved to Drive:', ZIP, f'{os.path.getsize(ZIP) / 1e6:.0f} MB', 'verified' if ok else 'VERIFY FAILED')
if ok:
    from google.colab import drive
    drive.flush_and_unmount(); print('Drive synced; disconnecting runtime'); runtime.unassign()